In [1]:
import pandas as pd
pd.options.mode.copy_on_write = False

columns = ['Date', 'Content']

df0 = pd.read_csv('crypto.csv', lineterminator='\n', usecols = columns)

df0.head()

,Date,Content
0,2022-11-30 11:53:21+00:00,#Ethereum price update: \n\n#ETH $1269.23 USD\...
1,2022-11-30 11:53:21+00:00,@mtmalinen @ecb Do well to understand that eve...
2,2022-11-30 11:53:21+00:00,Kayla #Ethereum Harold #世界杯 Egbert #百家乐 Lavern...
3,2022-11-30 11:53:20+00:00,#Bitcoin https://t.co/2koLlCvCri https://t.co/...
4,2022-11-30 11:53:18+00:00,"Yesterday we made a 3,000 SLAM buyback and loc..."


In [2]:
columns = ['date','text']
chunk = pd.read_csv('/kaggle/input/bitcoin-tweets/Bitcoin_tweets.csv',chunksize=100000,lineterminator='\n',usecols=columns)
df1 = pd.concat(chunk)

df1.head()

,date,text
0,2021-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...
1,2021-02-10 23:58:48,"😎 Today, that's this #Thursday, we will do a ""..."
2,2021-02-10 23:54:48,"Guys evening, I have read this article about B..."
3,2021-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...
4,2021-02-10 23:54:06,This network is secured by 9 508 nodes as of t...


In [3]:
columns = ['date', 'text']

df2 = pd.read_csv('/kaggle/input/bitcoin-tweets/Bitcoin_tweets_dataset_2.csv',engine="python", usecols=columns)

df2.head()

,date,text
0,2023-03-01 23:59:59,Which #bitcoin books should I think about read...
1,2023-03-01 23:59:47,"@ThankGodForBTC I appreciate the message, but ..."
2,2023-03-01 23:59:42,#Ethereum price update: \n\n#ETH $1664.02 USD\...
3,2023-03-01 23:59:36,CoinDashboard v3.0 is here\nAvailable on ios a...
4,2023-03-01 23:59:32,#Bitcoin Short Term Fractal (4H)💥\n\nIn lower ...


 ### Changing the data type of the Date columns of the 3 dataset to datetime data type to support grouping based on dates

In [4]:
df0['Date'] = pd.to_datetime(df0['Date'])
df1 = df1[pd.to_datetime(df1['date'], errors='coerce').notnull()]
df2 = df2[pd.to_datetime(df2['date'], errors='coerce').notnull()]

df2['date'] = pd.to_datetime(df2['date'])
df1['date'] = pd.to_datetime(df1['date'])

df2.info()
df1.info()
df0.info()

<class 'pandas.core.frame.DataFrame'>
Index: 169761 entries, 0 to 174396
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype         
---  ------  --------------   -----         
 0   date    169761 non-null  datetime64[ns]
 1   text    169761 non-null  object        
dtypes: datetime64[ns](1), object(1)
memory usage: 3.9+ MB
<class 'pandas.core.frame.DataFrame'>
Index: 4689288 entries, 0 to 4689353
Data columns (total 2 columns):
 #   Column  Dtype         
---  ------  -----         
 0   date    datetime64[ns]
 1   text    object        
dtypes: datetime64[ns](1), object(1)
memory usage: 107.3+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 2 columns):
 #   Column   Non-Null Count  Dtype              
---  ------   --------------  -----              
 0   Date     10000 non-null  datetime64[ns, UTC]
 1   Content  10000 non-null  object             
dtypes: datetime64[ns, UTC](1), object(1)
memory usage: 156.4+ KB


### Create Functions for pre-processing

In [5]:
import re
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /usr/share/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [6]:
def remove_hashtags(text):
    return re.sub(r'#\w+', '', text)

def remove_urls(text):
    return re.sub(r'http\S+|www\S+', '', text)

def remove_mentions(text):
    return re.sub(r'@\w+', '', text)

def remove_special_characters(text):
    return re.sub(r'[^a-zA-Z\s]', '', text)

### Call the functions to pre-process the Tweets column and store the pre-processed tweets to a seperate column

**Processed Done:**
* Remove Hashtags 
* Remove URL's 
* Remove Special Characters 
* Remove the '\n' in the tweets 

### Using the functions created pre-process each dataset and store the pre-processed data to a seperate column labeled "preprocessed_tweets"

In [7]:
df0['preprocessed_tweets'] = df0['Content'].apply(remove_hashtags)
df0['preprocessed_tweets'] = df0['preprocessed_tweets'].apply(remove_urls)
df0['preprocessed_tweets'] = df0['preprocessed_tweets'].apply(remove_mentions)
df0['preprocessed_tweets'] = df0['preprocessed_tweets'].apply(remove_special_characters)

df0['preprocessed_tweets'] = df0['preprocessed_tweets'].str.replace('\n', ' ')

df0.head()

,Date,Content,preprocessed_tweets
0,2022-11-30 11:53:21+00:00,#Ethereum price update: \n\n#ETH $1269.23 USD\...,price update USD Follow for recent ET...
1,2022-11-30 11:53:21+00:00,@mtmalinen @ecb Do well to understand that eve...,Do well to understand that every dip in the ...
2,2022-11-30 11:53:21+00:00,Kayla #Ethereum Harold #世界杯 Egbert #百家乐 Lavern...,Kayla Harold Egbert Laverna
3,2022-11-30 11:53:20+00:00,#Bitcoin https://t.co/2koLlCvCri https://t.co/...,
4,2022-11-30 11:53:18+00:00,"Yesterday we made a 3,000 SLAM buyback and loc...",Yesterday we made a SLAM buyback and locked i...


In [8]:
df1.dropna(subset=['text'], inplace=True)

df1['preprocessed_tweets'] = df1['text'].apply(remove_hashtags)
df1['preprocessed_tweets'] = df1['preprocessed_tweets'].apply(remove_urls)
df1['preprocessed_tweets'] = df1['preprocessed_tweets'].apply(remove_mentions)
df1['preprocessed_tweets'] = df1['preprocessed_tweets'].apply(remove_special_characters)

df1.head()

,date,text,preprocessed_tweets
0,2021-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...,Blue Ridge Bank shares halted by NYSE after A...
1,2021-02-10 23:58:48,"😎 Today, that's this #Thursday, we will do a ""...",Today thats this we will do a Take with ou...
2,2021-02-10 23:54:48,"Guys evening, I have read this article about B...",Guys evening I have read this article about BT...
3,2021-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...,BTC A big chance in a billion Price
4,2021-02-10 23:54:06,This network is secured by 9 508 nodes as of t...,This network is secured by nodes as of today...


In [9]:
df2.dropna(subset=['text'], inplace=True)

df2['preprocessed_tweets'] = df2['text'].apply(remove_hashtags)
df2['preprocessed_tweets'] = df2['preprocessed_tweets'].apply(remove_urls)
df2['preprocessed_tweets'] = df2['preprocessed_tweets'].apply(remove_mentions)
df2['preprocessed_tweets'] = df2['preprocessed_tweets'].apply(remove_special_characters)

df2.head()

,date,text,preprocessed_tweets
0,2023-03-01 23:59:59,Which #bitcoin books should I think about read...,Which books should I think about reading next
1,2023-03-01 23:59:47,"@ThankGodForBTC I appreciate the message, but ...",I appreciate the message but not a fan of the...
2,2023-03-01 23:59:42,#Ethereum price update: \n\n#ETH $1664.02 USD\...,price update \n\n USD\n \n\nFollow for rece...
3,2023-03-01 23:59:36,CoinDashboard v3.0 is here\nAvailable on ios a...,CoinDashboard v is here\nAvailable on ios and ...
4,2023-03-01 23:59:32,#Bitcoin Short Term Fractal (4H)💥\n\nIn lower ...,Short Term Fractal H\n\nIn lower timeframe p...


# Sentiment Analysis

### Get Sentiment scores using VADER and create a new column labeled as sentiment_scores to store the sentiment scores for each row

In [10]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

nltk.download('vader_lexicon')

sid = SentimentIntensityAnalyzer()
df0['sentiment_scores'] = df0['preprocessed_tweets'].apply(lambda x: sid.polarity_scores(x)['compound'])

df0.head()

/opt/conda/lib/python3.10/site-packages/nltk/twitter/__init__.py:20: UserWarning: The twython library has not been installed. Some functionality from the twitter package will not be available.
  warnings.warn("The twython library has not been installed. "


[nltk_data] Downloading package vader_lexicon to
[nltk_data]     /usr/share/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!


,Date,Content,preprocessed_tweets,sentiment_scores
0,2022-11-30 11:53:21+00:00,#Ethereum price update: \n\n#ETH $1269.23 USD\...,price update USD Follow for recent ET...,0.0000
1,2022-11-30 11:53:21+00:00,@mtmalinen @ecb Do well to understand that eve...,Do well to understand that every dip in the ...,0.5994
2,2022-11-30 11:53:21+00:00,Kayla #Ethereum Harold #世界杯 Egbert #百家乐 Lavern...,Kayla Harold Egbert Laverna,0.0000
3,2022-11-30 11:53:20+00:00,#Bitcoin https://t.co/2koLlCvCri https://t.co/...,,0.0000
4,2022-11-30 11:53:18+00:00,"Yesterday we made a 3,000 SLAM buyback and loc...",Yesterday we made a SLAM buyback and locked i...,-0.7695


In [11]:
df2['sentiment_scores'] = df2['preprocessed_tweets'].apply(lambda x: sid.polarity_scores(x)['compound'])

df2.head()

,date,text,preprocessed_tweets,sentiment_scores
0,2023-03-01 23:59:59,Which #bitcoin books should I think about read...,Which books should I think about reading next,0.0000
1,2023-03-01 23:59:47,"@ThankGodForBTC I appreciate the message, but ...",I appreciate the message but not a fan of the...,-0.1513
2,2023-03-01 23:59:42,#Ethereum price update: \n\n#ETH $1664.02 USD\...,price update \n\n USD\n \n\nFollow for rece...,0.0000
3,2023-03-01 23:59:36,CoinDashboard v3.0 is here\nAvailable on ios a...,CoinDashboard v is here\nAvailable on ios and ...,0.0000
4,2023-03-01 23:59:32,#Bitcoin Short Term Fractal (4H)💥\n\nIn lower ...,Short Term Fractal H\n\nIn lower timeframe p...,-0.2960


### For the large datset I'm doing the same thing where I get the sentiment scores in chunks of 100,000 like the creating of the dataframe of the first part of this notebook

> This part will take a some time and alot of CPU resources

In [12]:
chunk_size = 100000

for i in range(0, len(df1), chunk_size):
    chunk = df1.iloc[i:i+chunk_size].copy()

    chunk['sentiment_scores'] = chunk['preprocessed_tweets'].apply(lambda x: sid.polarity_scores(x)['compound'])

    df1.loc[chunk.index, 'sentiment_scores'] = chunk['sentiment_scores']


df1.head()

,date,text,preprocessed_tweets,sentiment_scores
0,2021-02-10 23:59:04,Blue Ridge Bank shares halted by NYSE after #b...,Blue Ridge Bank shares halted by NYSE after A...,0.2960
1,2021-02-10 23:58:48,"😎 Today, that's this #Thursday, we will do a ""...",Today thats this we will do a Take with ou...,0.4939
2,2021-02-10 23:54:48,"Guys evening, I have read this article about B...",Guys evening I have read this article about BT...,0.5719
3,2021-02-10 23:54:33,$BTC A big chance in a billion! Price: \487264...,BTC A big chance in a billion Price,0.2500
4,2021-02-10 23:54:06,This network is secured by 9 508 nodes as of t...,This network is secured by nodes as of today...,-0.2023


### Create functions to calculate sentiment scores and create threshholds for the different sentiment scores

In [13]:
def get_label(score):
    thresholds = {
        "Decrease": -0.5,
        "Slight Decrease": -0.1,
        "Neutral": 0,
        "Slight Increase": 0.5,
        "Increase": 1
    }

    for label, threshold in thresholds.items():
        if score <= threshold:
            sentiment_label = label
            break

    return sentiment_label

### Using the functions created use them for each dataframe and then grouping them by their dates to get the sentiment score and label for each day

In [14]:
df0['FormattedDate'] = df0['Date'].dt.strftime('%m-%d-%Y')

grouped_sentiments = df0.groupby('FormattedDate')['sentiment_scores'].mean().reset_index()

sorted_sentiments = grouped_sentiments.sort_values(by='FormattedDate', ascending=True)

for index, row in sorted_sentiments.iterrows():
    score = row['sentiment_scores']
    label = get_label(score)
    print(f"{row['FormattedDate']} : {row['sentiment_scores']:.2f} : {label}")

11-30-2022 : 0.12 : Slight Increase


In [15]:
df2['FormattedDate'] = df2['date'].dt.strftime('%m-%d-%Y')

grouped_sentiments = df2.groupby('FormattedDate')['sentiment_scores'].mean().reset_index()

sorted_sentiments = grouped_sentiments.sort_values(by='FormattedDate', ascending=True)

for index, row in sorted_sentiments.iterrows():
    score = row['sentiment_scores']
    label = get_label(score)
    print(f"{row['FormattedDate']} : {row['sentiment_scores']:.2f} : {label}")

02-25-2023 : 0.13 : Slight Increase
02-26-2023 : 0.18 : Slight Increase
02-27-2023 : 0.14 : Slight Increase
02-28-2023 : 0.14 : Slight Increase
03-01-2023 : 0.16 : Slight Increase
03-02-2023 : 0.13 : Slight Increase
03-03-2023 : 0.11 : Slight Increase
03-04-2023 : 0.14 : Slight Increase
03-05-2023 : 0.14 : Slight Increase


In [16]:
df1['FormattedDate'] = df1['date'].dt.strftime('%m-%d-%Y')

grouped_sentiments = df1.groupby('FormattedDate')['sentiment_scores'].mean().reset_index()

sorted_sentiments = grouped_sentiments.sort_values(by='FormattedDate', ascending=True)

for index, row in sorted_sentiments.iterrows():
    score = row['sentiment_scores']
    label = get_label(score)
    print(f"{row['FormattedDate']} : {row['sentiment_scores']:.2f} : {label}")

01-06-2023 : 0.17 : Slight Increase
01-07-2023 : 0.15 : Slight Increase
01-08-2023 : 0.14 : Slight Increase
01-09-2023 : 0.14 : Slight Increase
01-11-2022 : 0.22 : Slight Increase
01-12-2022 : 0.26 : Slight Increase
01-13-2022 : 0.23 : Slight Increase
01-14-2022 : 0.25 : Slight Increase
01-19-2022 : 0.20 : Slight Increase
01-20-2022 : 0.22 : Slight Increase
01-21-2022 : 0.15 : Slight Increase
01-22-2022 : 0.17 : Slight Increase
02-05-2021 : 0.13 : Slight Increase
02-06-2021 : 0.14 : Slight Increase
02-07-2021 : 0.14 : Slight Increase
02-08-2021 : 0.15 : Slight Increase
02-09-2021 : 0.15 : Slight Increase
02-09-2022 : 0.22 : Slight Increase
02-10-2021 : 0.16 : Slight Increase
02-13-2021 : 0.15 : Slight Increase
02-14-2021 : 0.25 : Slight Increase
02-14-2022 : 0.19 : Slight Increase
02-15-2021 : 0.32 : Slight Increase
02-15-2022 : 0.23 : Slight Increase
02-16-2022 : 0.21 : Slight Increase
02-17-2022 : 0.22 : Slight Increase
02-18-2021 : 0.16 : Slight Increase
02-18-2022 : 0.22 : Slight I

# **Conclusion**

> Since the dataset was taken during the period of 2021-2023 where Bitcoin prices was steadily increasing everyday all sentiments for the dataset timeframe shows slight increase and after the time frame on the dataset the price kept increasing steadily without sudden drops or increase in price.